# Prepare DS2_wide feature table

This notebook converts `DS2_full.xlsx` dataset into the widened feature layout of the D3 dataset used in Paper IV.

The output is a `DS2_wide` table with:
- 22 input features
- 2 tensile-property targets
- column names aligned with the D3 schema
- fixed neutral anchors for missing pre-aging and final-cycle segments

The purpose is schema harmonization, not new data generation.

Paper IV: 
*Øien, Christian Dalheim. Predicting mechanical properties of Al–Mg–Si extrusions
containing emulated post-consumer scrap content using hybrid modeling.
Engineering Proceedings. (In review, March 2026)*

## 1. Setup

Define input and output paths, the expected wide-format feature columns, and the target columns.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

DS2_FULL = Path("../paper_II/data/DS2_full.xlsx")
DS2_OUT = DATA_DIR / "DS2_wide.xlsx"
D3_FILE = DATA_DIR / "D3.xlsx"

FEATURE_COLS = [
    "Mg", "Si", "Fe", "Mn", "Cu", "Cr",
    "T4", "T5", "T6", "T7", "T8", "T9", "T10", "T11",
    "t4_5", "t5_6", "t6_7", "t7_8", "t8_9", "t9_10", "t10_11",
    "Scheil",
]
TARGET_COLS = ["Rp0.2 [MPa]", "Rm [MPa]"]

## 2. Convert DS2 to wide format

Load `DS2_full.xlsx`, rename chemistry columns, expand the aging-cycle representation to the wide schema, and save it as `DS2_wide.xlsx`

Since DS2 contains only the artificial-aging segment, fixed neutral values are inserted for the additional temperature and duration anchors required by the D3 layout.

In [2]:
ds2_full = pd.read_excel(DS2_FULL)

ds2_wide = pd.DataFrame()
for old, new in {
    "Mg [wt.%]": "Mg",
    "Si [wt.%]": "Si",
    "Fe [wt.%]": "Fe",
    "Mn [wt.%]": "Mn",
    "Cu [wt.%]": "Cu",
    "Cr [wt.%]": "Cr",
}.items():
    ds2_wide[new] = ds2_full[old]

# DS2_full contains only the artificial-ageing segment T6-T10.
# Fixed neutral pre-ageing/final anchors widen it to the Paper IV D3 schema.
ds2_wide["T4"] = 560
ds2_wide["T5"] = 20
for col in ["T6", "T7", "T8", "T9", "T10"]:
    ds2_wide[col] = ds2_full[col]
ds2_wide["T11"] = ds2_full["T10"]

ds2_wide["t4_5"] = 10
ds2_wide["t5_6"] = 1
for col in ["t6_7", "t7_8", "t8_9", "t9_10"]:
    ds2_wide[col] = ds2_full[col]
ds2_wide["t10_11"] = 1

ds2_wide["Scheil"] = ds2_full["Scheil"]
ds2_wide["Rp0.2 [MPa]"] = ds2_full["Rp0.2 [MPa]"]
ds2_wide["Rm [MPa]"] = ds2_full["Rm [MPa]"]
ds2_wide = ds2_wide[FEATURE_COLS + TARGET_COLS]

ds2_wide.to_excel(DS2_OUT, index=False)
print(f"Wrote {DS2_OUT} with shape {ds2_wide.shape}")
display(ds2_wide)

Wrote data\DS2_wide.xlsx with shape (2895, 24)


,Mg,Si,Fe,Mn,Cu,Cr,T4,T5,T6,T7,...,t4_5,t5_6,t6_7,t7_8,t8_9,t9_10,t10_11,Scheil,Rp0.2 [MPa],Rm [MPa]
0,0.435,0.485,0.214,0.206,0.042,0.0,560,20,20,150,...,10,1,600,60,480,60,1,0.009686,129.0,192.5
1,0.435,0.485,0.214,0.206,0.042,0.0,560,20,20,155,...,10,1,600,120,540,60,1,0.012190,154.3,208.5
2,0.435,0.485,0.214,0.206,0.042,0.0,560,20,20,160,...,10,1,600,300,300,60,1,0.013489,136.2,197.4
3,0.435,0.485,0.214,0.206,0.042,0.0,560,20,20,150,...,10,1,600,60,480,120,1,0.016255,156.6,210.6
4,0.435,0.485,0.214,0.206,0.042,0.0,560,20,20,155,...,10,1,600,120,300,120,1,0.019008,149.8,219.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2890,0.549,0.646,0.204,0.060,0.013,0.0,560,20,20,160,...,10,1,600,600,345600,1800,1,14.402935,229.3,295.1
2891,0.549,0.646,0.204,0.060,0.013,0.0,560,20,20,160,...,10,1,600,1800,210540,3600,1,15.394566,226.8,293.7
2892,0.549,0.646,0.204,0.060,0.013,0.0,560,20,20,190,...,10,1,7200,259020,60,60,1,20.850021,227.9,289.3
2893,0.549,0.646,0.204,0.060,0.013,0.0,560,20,20,175,...,10,1,600,1800,206520,600,1,25.391084,211.3,275.9


## 3. Check schema compatibility

If `D3.xlsx` is available, compare its column order with the generated `DS2_wide` table.

This verifies that both datasets can be used by the same downstream modelling notebooks.

In [3]:
if D3_FILE.exists():
    d3 = pd.read_excel(D3_FILE)
    print((d3.columns == ds2_wide.columns).all())
else:
    print(f"{D3_FILE} not found.")

True
